In [1]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../../config"

# Collections

The Spotify service implements the standard collection interfaces that are common across all services.

## Library Collection

The {py:class}`SpotifyLibraryCollection <plistsync.services.spotify.SpotifyLibraryCollection>` represents the full spotify catalog from the perspective of an authenticated user.

In [3]:
from plistsync.services.spotify import SpotifyLibraryCollection

library = SpotifyLibraryCollection()

### Track Lookup

The Spotify library collection supports lookup by global IDs, specifically by `spotify_id` and `isrc`. This allows for efficient retrieval of tracks using these identifiers.

In [12]:
# By spotify id
if track1 := library.find_by_global_ids({"spotify_id": "6fzwardfFs6sVfNA5R1ypt"}):
    print(track1)

# By isrc
if track2 := library.find_by_global_ids({"isrc": "GBBZH9601601"}):
    print(track2)

assert track1 == track2  # TODO!

Track[Origin Unknown > Valley of the Shadows, 8159085718064]
Track[Origin Unknown > Valley of the Shadows, 8159085627318]


AssertionError: 

If you need/want to lookup many tracks at once, consider using {py:meth}`SpotifyLibraryCollection.find_many_by_global_ids <plistsync.services.spotify.SpotifyLibraryCollection.find_many_by_global_ids>` for better performance.

## Playlist Collection

The {py:class}`SpotifyPlaylistCollection <plistsync.services.spotify.SpotifyPlaylistCollection>` represents a playlist of an authenticated user. 

### Retrieving playlists

You can retrieve all playlists of a user using the library's {py:meth}`SpotifyLibraryCollection.playlists <plistsync.services.spotify.SpotifyLibraryCollection.playlists>` property.

In [ ]:
# This may take a while if you have many playlists!
playlists = library.playlists
for playlist in playlists:
    print(f"Playlist: {playlist.name} ({len(playlist.tracks)} tracks)")

If you just want a specific playlist, you can use the library's {py:meth}`SpotifyLibraryCollection.get_playlist <plistsync.services.spotify.SpotifyLibraryCollection.get_playlist>` method to retriev a playlist by name or id.

In [ ]:
# By name
if pl := library.get_playlist("Tribal Dnb"):
    print(f"Found playlist by name: {pl.name} ({len(pl.tracks)} tracks)")
else:
    print("Playlist not found by name.")

# TODO: By id/url/uri
# This should also fix the type checking issues below
# get by id should raise if not found
# pl = library.get_playlist_by_id("spotify:playlist:37i9dQZF1DX8tZsk68tuDw")

### Creating playlists

Currently we do not support creating new playlists with a unified interface. We will reevaluate this at a later point in time. For now you need to use the Spotify API directly to create new playlists.

In [ ]:
from plistsync.services.spotify import SpotifyPlaylistCollection

pl_data = library.api.playlist.create(
    name="My New Playlist",
    description="Created via plistsync",
)
pl = SpotifyPlaylistCollection.from_response_data(
    library,
    pl_data,
)

### Updating a playlist

For updating a playlist, you should use the playlist's {py:meth}`SpotifyPlaylistCollection.edit <plistsync.services.spotify.SpotifyPlaylistCollection.edit>` context manager. This ensures that all changes are properly saved back to Spotify when you exit the context. This also minifies changes and therefore reduces API calls.


In [18]:
# Updating a playlist description/name
with pl.edit():
    pl.description = "Top DNB song und so"

pl.description

'Top DNB song und so'

In [ ]:
# Add a track to a playlist
new_track = library.find_by_global_ids({"isrc": "GBBZH9601601"})

with pl.edit():
    pl.tracks.append(new_track)  # TODO: SpotifyTrack vs SpotifyPlaylistTrack

In [ ]:
# Reordering tracks in a playlist
with pl.edit():
    pl.tracks.insert(15, pl.tracks.pop())